# IBL ONE integration walkthrough

This notebook shows how to:

1. connect to the public International Brain Laboratory Alyx server with `ONE`
2. search subjects and sessions
3. load behavioural trials across the main public IBL storage layouts
4. convert those trials into the standard psytrax data dict
5. run a small example fit on the converted data

Install the required extras first:

```bash
pip install -e .[web,ibl]
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from one.api import ONE

import psytrax
from psytrax.models.logistic import (
    log_lik_trial,
    N_PARAMS,
    PARAM_NAMES,
    default_E0,
    default_hyper,
)


In [ ]:
IBL_BASE_URL = "https://openalyx.internationalbrainlab.org"
IBL_PUBLIC_USER = "intbrainlab"
IBL_PUBLIC_PASSWORD = "international"

one = ONE(
    base_url=IBL_BASE_URL,
    username=IBL_PUBLIC_USER,
    password=IBL_PUBLIC_PASSWORD,
    silent=True,
)

one


## Subject and session discovery

The Streamlit app uses an autocomplete-style subject search backed by Alyx `subjects` queries. We can reproduce the same pattern here.


In [ ]:
def search_subjects(query, limit=25):
    query = query.strip()
    if len(query) < 2:
        return []

    matches = one.alyx.rest(
        "subjects",
        "list",
        django=f"nickname__istartswith,{query}",
        limit=limit,
    )
    matches = matches[:limit] if not isinstance(matches, list) else matches
    if not matches:
        matches = one.alyx.rest(
            "subjects",
            "list",
            django=f"nickname__icontains,{query}",
            limit=limit,
        )
        matches = matches[:limit] if not isinstance(matches, list) else matches
    return list(dict.fromkeys(rec["nickname"] for rec in matches))


def search_sessions(subject):
    eids, details = one.search(subject=subject, details=True)
    sessions = []
    for eid, det in zip(eids, details):
        if det.get("subject") != subject:
            continue
        sessions.append(
            {
                "eid": str(det.get("id", eid)),
                "date": str(det.get("date") or det.get("start_time", "")[:10] or eid),
                "lab": det.get("lab", "?"),
                "number": det.get("number", "?"),
                "task_protocol": det.get("task_protocol", "?"),
            }
        )
    sessions.sort(key=lambda s: s["date"])
    return sessions


SUBJECT_QUERY = "KS023"
subject_matches = search_subjects(SUBJECT_QUERY)
subject_matches[:10]


In [ ]:
SUBJECT = next((s for s in subject_matches if s == SUBJECT_QUERY), subject_matches[0])
sessions = search_sessions(SUBJECT)
sessions_df = pd.DataFrame(sessions)
sessions_df.head(10)


## Loading trials across IBL layouts

Public IBL sessions may expose trials as:

- an assembled `trials` object
- a parquet table (`_ibl_trials.table.pqt`)
- older per-field arrays such as `_ibl_trials.choice.npy`

The helpers below mirror the app's fallback order.


In [ ]:
def load_ibl_trials(eid):
    errors = []

    for kwargs in (
        {},
        {"collection": "alf"},
        {"namespace": "ibl"},
        {"namespace": "ibl", "collection": "alf"},
    ):
        try:
            return one.load_object(eid, "trials", **kwargs)
        except Exception as exc:
            label = "load_object" if not kwargs else f"load_object{kwargs}"
            errors.append(f"{label}: {exc}")

    try:
        table = one.load_dataset(eid, "_ibl_trials.table.pqt")
        if isinstance(table, pd.DataFrame):
            return table
    except Exception as exc:
        errors.append(f"load_dataset: {exc}")

    try:
        table_path = one.load_dataset(eid, "_ibl_trials.table.pqt", download_only=True)
        return pd.read_parquet(table_path)
    except Exception as exc:
        errors.append(f"read_parquet: {exc}")

    required = {
        "choice": "_ibl_trials.choice.npy",
        "contrastLeft": "_ibl_trials.contrastLeft.npy",
        "contrastRight": "_ibl_trials.contrastRight.npy",
        "response_times": "_ibl_trials.response_times.npy",
    }
    optional = {
        "feedbackType": "_ibl_trials.feedbackType.npy",
        "rewardVolume": "_ibl_trials.rewardVolume.npy",
        "probabilityLeft": "_ibl_trials.probabilityLeft.npy",
        "stimOn_times": "_ibl_trials.stimOn_times.npy",
        "feedback_times": "_ibl_trials.feedback_times.npy",
        "goCue_times": "_ibl_trials.goCue_times.npy",
        "firstMovement_times": "_ibl_trials.firstMovement_times.npy",
        "intervals": "_ibl_trials.intervals.npy",
    }

    trial_dict = {}
    missing_required = []
    for key, dataset in required.items():
        try:
            trial_dict[key] = one.load_dataset(eid, dataset)
        except Exception as exc:
            missing_required.append(dataset)
            errors.append(f"manual {dataset}: {exc}")

    if not missing_required:
        for key, dataset in optional.items():
            try:
                trial_dict[key] = one.load_dataset(eid, dataset)
            except Exception:
                pass
        return trial_dict

    raise RuntimeError("; ".join(errors))


def trial_array(trials, key):
    if isinstance(trials, pd.DataFrame):
        if key in trials.columns:
            return np.asarray(trials[key], dtype=float)
        return None
    if hasattr(trials, key):
        return np.asarray(getattr(trials, key), dtype=float)
    if isinstance(trials, dict) and key in trials:
        return np.asarray(trials[key], dtype=float)
    return None


def relative_rt_candidates(n_trials, anchor, event):
    if anchor is None or event is None:
        return np.full(n_trials, np.nan)
    delta = np.asarray(event, dtype=float) - np.asarray(anchor, dtype=float)
    valid = np.isfinite(delta) & (delta > 0)
    return np.where(valid, delta, np.nan)


def looks_like_relative_rt(raw_rt):
    raw_rt = np.asarray(raw_rt, dtype=float)
    finite = raw_rt[np.isfinite(raw_rt)]
    if finite.size == 0:
        return False
    if np.nanpercentile(finite, 95) <= 20:
        return True
    if finite.size < 3:
        return False
    return np.mean(np.diff(finite) >= 0) < 0.95


def ibl_rt(trials):
    response_times = trial_array(trials, "response_times")
    if response_times is None:
        raise KeyError("response_times")

    n_trials = len(response_times)
    stim_on = trial_array(trials, "stimOn_times")
    first_move = trial_array(trials, "firstMovement_times")
    go_cue = trial_array(trials, "goCue_times")

    rt = np.full(n_trials, np.nan)
    for candidate in (
        relative_rt_candidates(n_trials, stim_on, first_move),
        relative_rt_candidates(n_trials, stim_on, response_times),
        relative_rt_candidates(n_trials, go_cue, first_move),
        relative_rt_candidates(n_trials, go_cue, response_times),
    ):
        use = np.isnan(rt) & np.isfinite(candidate)
        rt[use] = candidate[use]

    if looks_like_relative_rt(response_times):
        use = np.isnan(rt) & np.isfinite(response_times) & (response_times > 0)
        rt[use] = response_times[use]

    return rt


def ibl_to_psytrax(trials_list):
    all_c, all_r, all_t, all_reward, all_p_left = [], [], [], [], []
    session_lengths = []

    for trials in trials_list:
        c_left = trial_array(trials, "contrastLeft")
        c_right = trial_array(trials, "contrastRight")
        c_left = np.nan_to_num(c_left, nan=0.0)
        c_right = np.nan_to_num(c_right, nan=0.0)
        signed_contrast = c_right - c_left

        choice = trial_array(trials, "choice")
        responses = np.where(choice == -1.0, 1.0, np.where(choice == 1.0, 0.0, np.nan))

        feedback = trial_array(trials, "feedbackType")
        if feedback is not None:
            reward = np.where(feedback == -1.0, 0.0, np.where(feedback == 1.0, 1.0, np.nan))
        else:
            reward_volume = trial_array(trials, "rewardVolume")
            reward = (reward_volume > 0).astype(float)

        rt = ibl_rt(trials)
        p_left = trial_array(trials, "probabilityLeft")
        if p_left is None:
            p_left = np.full(len(signed_contrast), 0.5)

        valid = (
            np.isfinite(signed_contrast)
            & np.isfinite(responses)
            & np.isfinite(rt)
            & np.isfinite(reward)
            & np.isfinite(p_left)
            & (rt > 0)
        )

        all_c.append(signed_contrast[valid])
        all_r.append(responses[valid])
        all_t.append(rt[valid])
        all_reward.append(reward[valid])
        all_p_left.append(p_left[valid])
        session_lengths.append(int(np.sum(valid)))

    return {
        "inputs": {
            "c": np.concatenate(all_c),
            "reward": np.concatenate(all_reward),
            "p_left": np.concatenate(all_p_left),
        },
        "responses": np.concatenate(all_r),
        "times": np.concatenate(all_t),
        "session_lengths": np.array(session_lengths, dtype=int),
    }


In [ ]:
N_SESSIONS = 4
selected_sessions = sessions[:N_SESSIONS]
trial_objects = [load_ibl_trials(sess["eid"]) for sess in selected_sessions]
raw = ibl_to_psytrax(trial_objects)

print(f"Loaded {len(raw['responses'])} trials across {len(raw['session_lengths'])} sessions")
pd.DataFrame(
    {
        "signed_contrast": raw["inputs"]["c"][:10],
        "response_is_right": raw["responses"][:10],
        "rt_s": raw["times"][:10],
        "reward": raw["inputs"]["reward"][:10],
        "p_left": raw["inputs"]["p_left"][:10],
    }
)


## Fit a small example model

For a quick end-to-end demonstration we use the built-in logistic model. You can swap this for a race or DDM model once the data-loading step looks right.


In [ ]:
result = psytrax.fit(
    data=raw,
    log_lik_trial=log_lik_trial,
    n_params=N_PARAMS,
    param_names=PARAM_NAMES,
    hyper=default_hyper(),
    E0=default_E0(len(raw["responses"])),
    shared_sigma=True,
    session_boundaries=True,
    subject_name=f"{SUBJECT}_ibl_demo",
    save=False,
    verbose=True,
)

result["log_evidence"]


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(result["params"][0], label=PARAM_NAMES[0], color="#4e9af1")
axes[1].plot(result["params"][1], label=PARAM_NAMES[1], color="#f1a44e")
axes[0].set_ylabel(PARAM_NAMES[0])
axes[1].set_ylabel(PARAM_NAMES[1])
axes[1].set_xlabel("Trial")
axes[0].set_title(f"{SUBJECT}: logistic fit on {len(raw['responses'])} IBL trials")
for ax in axes:
    ax.grid(True, color="#ececec")
plt.tight_layout()
